## Главная задача

Из накопительных показаний получить **суточное потребление** (расход за день =
разница показаний между соседними днями) и найти в потреблении аномалии.

## Суточные потребления

In [34]:
import pandas as pd

df_pokazaniya = pd.read_csv(
    "pokazaniya.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"anon_id": str},
    parse_dates=["data"],
)

df_pokazaniya = df_pokazaniya.sort_values(["anon_id", "data"])

df_pokazaniya["dni_mezhdu"] = df_pokazaniya.groupby("anon_id")["data"].diff().dt.days
df_pokazaniya["raznica"] = df_pokazaniya.groupby("anon_id")["pokazanie"].diff().round(3)

# Расходом именно за одни сутки считаем разницу только соседних дат.
df_pokazaniya["rashod_za_sutki"] = df_pokazaniya["raznica"].where(df_pokazaniya["dni_mezhdu"] == 1)

In [35]:
df_pokazaniya.to_csv('pokazanaiya_res.csv', sep=";", index=False, encoding="utf-8-sig")

## Что найти (аномалии потребления)

Для каждого пункта: ваш критерий (порог) + сколько приборов + список/примеры
самых ярких `anon_id`.

## 1. **Отрицательный расход** — показание уменьшилось (счётчик «пошёл назад»).
   Что это может означать?
   накопительное показание обычно должно увеличиваться или оставаться прежним. Его уменьшение может быть связано со сбросом счётчика, заменой прибора, ошибкой передачи данных, неисправностью или обратным движением счётчика. По отрицательной разнице нельзя определить точную причину — нужно проверить события «сброс» и историю прибора.

In [38]:
# Строки, где показание уменьшилось
df_minus_rasxod = df_pokazaniya[
    df_pokazaniya["raznica"] < 0
]

# Количество уникальных приборов
print("Приборов:", df_minus_rasxod["anon_id"].nunique())

# Количество случаев уменьшения
print("Случаев:", len(df_minus_rasxod))

# Количество падений показаний у каждого прибора
top_minus = (
    df_minus_rasxod.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_padeniy")
)

print('В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений')
display(top_minus.to_frame())



Приборов: 102
Случаев: 269
В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений


,kolichestvo_padeniy
anon_id,
2030325,13
7931977,12
3917499,10
8190741,10
7293232,8


## 2. **Всплески** — аномально большой расход за сутки. Дайте и абсолютный порог,
   и относительный (например, в N раз больше обычного для этого прибора).

In [37]:
# 10 самых больших падений
print(
    df_minus_rasxod.nsmallest(10, "raznica")[
        ["anon_id", "data", "pokazanie", "raznica"]
    ]
)

        anon_id       data   pokazanie      raznica
438368  2519659 2026-06-09   72.726000 -4273492.461
438370  2519659 2026-09-25   72.726000 -4273492.461
442978  2491162 2026-07-10  380.814286 -3006486.903
455297  8908023 2026-09-08    1.049375 -2684366.742
397635  2159987 2025-10-19  315.120000    -2949.060
266444  7891133 2025-10-19  236.581000    -2293.389
163197  8190741 2026-02-06  615.774000    -1311.426
415976  1423304 2026-09-13  486.252750    -1310.846
398281  7293232 2026-04-17  697.100000    -1310.800
146302  5704914 2026-03-01  335.267000    -1310.783


## 3

## 4

## 5

##  6

## 7

## 8